In [12]:
import pandas as pd
import csv
import sys
from pathlib import Path
import requests
from bs4 import BeautifulSoup


In [7]:
dataset_path = ".." / Path.cwd().parent / "dataset" / "echr_2_0_0_unstructured_cases.json"
df = pd.read_json(dataset_path)

In [8]:
def extract_section_markdown(case_data, section_name):
    """
    Extracts a specific section from the case data and formats it as markdown.

    Args:
        case_data (dict): The dictionary containing case data.
        section_name (str): The name of the section to extract (e.g., 'law', 'facts').

    Returns:
        str: Formatted markdown text of the specified section.
    """
    
    section_text = ""
    if 'content' in case_data:
        documents = case_data['content']
        for key, sections in documents.items():
            for section in sections:
                if section.get('section_name', '').upper() == section_name.upper():
                    # Append the section title with a heading format
                    section_text += f"### {section['content']}\n\n"
                    section_text += extract_elements(section['elements'])
                    section_text += "\n"

    return section_text.strip()

def extract_elements(elements, indent=0):
    content_text = ""
    for element in elements:
        # Add indentation for each level of nesting
        prefix = "  " * indent
        content_text += f"{prefix}- {element['content']}\n"
        if 'elements' in element:
            # Recursively add further nested elements with increased indentation
            content_text += extract_elements(element['elements'], indent + 1)
    return content_text

In [9]:
df['law'] = df.apply(lambda x: extract_section_markdown(x, 'law'), axis = 1)
df['facts'] = df.apply(lambda x: extract_section_markdown(x, 'facts'), axis = 1)
df['the_conclusion'] = df.apply(lambda x: extract_section_markdown(x, 'conclusion'), axis = 1)

In [13]:
def get_full_text_from_html(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    # Remove script and style elements
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    text = soup.get_text()
    
    # Replace multiple whitespace characters with a single space
    text = ' '.join(text.split())

    # Replace non-breaking space characters with regular spaces
    text = text.replace(u'\xa0', u' ')
        
    return text


In [14]:
BASE_URL = 'https://hudoc.echr.coe.int/app/conversion/docx/html/body?library=ECHR&id='

def full_text_extraction(dataframe):
    text_list = []
    for itemid in dataframe['itemid']:
        r = requests.get(BASE_URL + itemid, timeout=5)
        text_list.append(get_full_text_from_html(r.text))
    return text_list


In [ ]:
text_list = full_text_extraction(df)

In [15]:
filename_path = ".." / Path.cwd().parent / "dataset" / "text_list.csv"
text_list= []

# Set the maximum field size to a higher limit, e.g., the maximum integer size
csv.field_size_limit(sys.maxsize)

with open(filename_path, 'r', newline='', encoding='utf-8') as file:
    reader = csv.reader(file)
    for row in reader:
        if row:  # This checks if the row is not empty
            full_text = ' '.join(row)
            text_list.append(full_text)  


In [16]:
df['full_text'] = text_list

In [17]:
df['judgement_type_1'] = df['documentcollectionid'].apply(lambda x: x[1])
df['judgement_type_2'] = df['documentcollectionid'].apply(lambda x: x[2])

In [18]:
df

,__articles,__conclusion,_decision_body,applicability,application,appno,article,attachments,conclusion,content,...,respondent,scl,separateopinion,typedescription,law,facts,the_conclusion,full_text,judgement_type_1,judgement_type_2
0,13;13+P1-1;P1-1,Violation of Article 1 of Protocol No. 1 - Pro...,"Françoise Tulkens, President,\n\tVladimiro Zag...",,MS WORD,33888/05,"[p1-1, 13]",{'001-95845.docx': {}},"[{'article': 'p1-1', 'base_article': 'p1-1', '...","{'001-95845.docx': [{'content': 'PROCEDURE', '...",...,SRB,[],FALSE,15,### THE LAW\n\n- I. THE APPLICANT'S DEATH\n ...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",SECOND SECTION CASE OF МILICA POPOVIĆ v. SERBI...,JUDGMENTS,CHAMBER
1,10;10-1;10-2;35;35-3-b;41,Preliminary objection dismissed (Art. 35) Admi...,"Tim Eicke, President,\n\tYonko Grozev,\n\tFari...",,MS WORD,10783/14,"[35, 41, 10]",{'001-209033.docx': {}},"[{'details': ['Art. 35'], 'element': 'Prelimin...",{'001-209033.docx': [{'content': 'INTRODUCTION...,...,BGR,"[Albert-Engelmann-Gesellschaft mbH v. Austria,...",TRUE,15,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLE ...,### THE FACTS\n\n- 2. The applicant was born ...,"### FOR THESE REASONS, THE COURT\n\n- Declares...",FOURTH SECTIONCASE OF HANDZHIYSKI v. BULGARIA(...,JUDGMENTS,CHAMBER
2,5;5-4;8;8-1;8-2;35;35-1;35-3-a;37;37-1-c;41,Preliminary objections dismissed (Art. 35) Adm...,"Helena Jäderblom, President,\n\tBranko Lubarda...",,MS WORD,29431/05;7070/06;5402/07,"[41, 8, 5, 35, 37]",{'001-178343.docx': {'table-0': [{'Applicant’s...,"[{'details': ['Art. 35'], 'element': 'Prelimin...","{'001-178343.docx': [{'content': 'PROCEDURE', ...",...,RUS,"[Ahtinen v. Finland (dec.), no. 48907/99, 31 M...",TRUE,15,### THE LAW\n\n- I. JOINDER OF THE APPLICATIO...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",THIRD SECTION CASE OF ZUBKOV AND OTHERS v. RUS...,JUDGMENTS,CHAMBER
3,6;6-1,Violation of Art. 6-1,"Nicolas Bratza, President,\n\tJosep Casadevall...",,MS WORD,11101/04,[6],{'001-84572.docx': {}},"[{'article': '6-1', 'base_article': '6', 'elem...","{'001-84572.docx': [{'content': 'PROCEDURE', '...",...,POL,[],FALSE,15,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",FOURTH SECTION CASE OF RYGALSKI v. POLAND (App...,JUDGMENTS,CHAMBER
4,3;8;8-1,Violation of Article 3 - Prohibition of tortur...,"Nina Vajić, President,\n\tAnatoly Kovler,\n\tP...",,MS WORD,19433/07,"[3, 8]",{'001-112534.docx': {}},"[{'article': '3', 'base_article': '3', 'detail...","{'001-112534.docx': [{'content': 'PROCEDURE', ...",...,RUS,[],FALSE,15,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",FIRST SECTION CASE OF TYAGUNOVA v. RUSSIA (App...,JUDGMENTS,CHAMBER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16091,3,Violation of Art. 3 (substantive aspect),"Josep Casadevall, President,\n\tCorneliu Bîrsa...",,MS WORD,31725/04,[3],{'001-106551.docx': {}},"[{'article': '3', 'base_article': '3', 'detail...","{'001-106551.docx': [{'content': 'PROCEDURE', ...",...,ROU,[],FALSE,15,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",THIRD SECTION CASE OF BĂDILĂ v. ROMANIA (Appli...,JUDGMENTS,CHAMBER
16092,5;5-1;5-4;13+3;13;3,Violation of Art. 5-1;Violation of Art. 5-4;Vi...,"Christos Rozakis, President,\n\tNina Vajić,\n\...",,MS WORD,54219/08,"[3, 5, 13]",{'001-100198.docx': {}},"[{'article': '5-1', 'base_article': '5', 'elem...","{'001-100198.docx': [{'content': 'PROCEDURE', ...",...,RUS,[],FALSE,15,### THE LAW\n\n- I. THE GOVERNMENT'S PRELIMIN...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",FIRST SECTION CASE OF KARIM

In [42]:
df['__articles']

0                                    13;13+P1-1;P1-1
1                          10;10-1;10-2;35;35-3-b;41
2        5;5-4;8;8-1;8-2;35;35-1;35-3-a;37;37-1-c;41
3                                              6;6-1
4                                            3;8;8-1
                            ...                     
16091                                              3
16092                            5;5-1;5-4;13+3;13;3
16093                                              6
16094                                              3
16095                                           3;34
Name: __articles, Length: 16096, dtype: object

In [34]:
echr_df = df[['itemid', 'docname','article','appno','judgementdate', 'law', 'facts', 'the_conclusion', 'full_text', 'respondent', 'judgementdate']]

In [38]:
foe_echr_df = echr_df[echr_df['article'].apply(lambda x: '10' in x)]
foe_echr_df = foe_echr_df.reset_index(drop=True)
foe_echr_df

,itemid,docname,article,appno,judgementdate,law,facts,the_conclusion,full_text,respondent,judgementdate
0,001-209033,CASE OF HANDZHIYSKI v. BULGARIA,"[35, 41, 10]",10783/14,06/04/2021 00:00:00,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLE ...,### THE FACTS\n\n- 2. The applicant was born ...,"### FOR THESE REASONS, THE COURT\n\n- Declares...",FOURTH SECTIONCASE OF HANDZHIYSKI v. BULGARIA(...,BGR,06/04/2021 00:00:00
1,001-57513,CASE OF KOSIEK v. GERMANY,[10],9704/82,28/08/1986 00:00:00,### AS TO THE LAW\n\n- I. THE GOVERNMENT’S P...,### AS TO THE FACTS\n\n- 11. Mr. Rolf Kosiek...,"### FOR THESE REASONS, THE COURT\n\n- Holds by...",COURT (PLENARY) CASE OF KOSIEK v. GERMANY (App...,DEU,28/08/1986 00:00:00
2,001-155196,CASE OF MEHDIYEV v. AZERBAIJAN,"[41, 3, 5, 35, 10]",59075/09,18/06/2015 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT\n\n- 1. Decl...",FIRST SECTION CASE OF MEHDIYEV v. AZERBAIJAN (...,AZE,18/06/2015 00:00:00
3,001-84268,CASE OF FEVZİ SAYGILI v. TURKEY,"[14, 10, 13]",74243/01,08/01/2008 00:00:00,### THE LAW\n\n- I. THE GOVERNMENT'S PRELIMIN...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",SECOND SECTION CASE OF FEVZİ SAYGILI v. TURKEY...,TUR,08/01/2008 00:00:00
4,001-107591,CASE OF KILIÇ AND EREN v. TURKEY,[10],43807/07,29/11/2011 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",SECOND SECTION CASE OF KILIÇ AND EREN v. TURKE...,TUR,29/11/2011 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...
964,001-201087,CASE OF RELIGIOUS COMMUNITY OF JEHOVAH'S WITNE...,[10],52884/09,20/02/2020 00:00:00,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLEs...,### THE FACTS\n\n- THE CIRCUMSTANCES OF THE CA...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",FIFTH SECTIONCASE OF RELIGIOUS COMMUNITY OF JE...,AZE,20/02/2020 00:00:00
965,001-58032,"CASE OF X, Y AND Z v. THE UNITED KINGDOM","[8, 10, 14]",21830/93,22/04/1997 00:00:00,### AS TO THE LAW\n\n- I. ALLEGED VIOLATION ...,### AS TO THE FACTS\n\n- I. Circumstances of...,"### FOR THESE REASONS, THE COURT\n\n- 1. Hol...","COURT (GRAND CHAMBER) CASE OF X, Y AND Z v. TH...",GBR,22/04/1997 00:00:00
966,001-217373,CASE OF ZAO INFORMATSIONNOYE AGENTSTVO ROSBALT...,[10],16503/14,24/05/2022 00:00:00,### APPLICATION OF ARTICLE 41 OF THE CONVENTIO...,,### THE COURT’S ASSESSMENT\n\n- ALLEGED VIOLAT...,THIRD SECTIONCASE OF ZAO INFORMATSIONNOYE AGEN...,RUS,24/05/2022 00:00:00
967,001-186008,CASE OF ÇETİN AND GEDİK v. TURKEY,[10],29899/07;33333/08,04/09/2018 00:00:00,### THE LAW\n\n- I. JOINDER OF THE APPLICATIO...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",SECOND SECTION CASE OF ÇETİN AND GEDİK v. TURK...,TUR,04/09/2018 00:00:00


In [40]:
foe_echr_df.columns

Index(['itemid', 'docname', 'article', 'appno', 'judgementdate', 'law',
       'facts', 'the_conclusion', 'full_text', 'respondent', 'judgementdate'],
      dtype='object')

In [39]:
csv_path = ".." / Path.cwd().parent / "dataset" / "foe_echr.csv"
foe_echr_df.to_csv(csv_path)